# 03 - Profile-Aware Span-Completion Training

Reuse the span-completion profile-aware tokenizer artifacts from notebook 02, then continue training from the existing protein checkpoint.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import yaml


def find_repo_dir_for_import(start: Path) -> Path:
    candidates: list[Path] = []
    env_value = os.environ.get("MDNAC_REPO_DIR")
    if env_value:
        candidates.append(Path(env_value).expanduser())
    resolved_start = start.expanduser().resolve()
    candidates.extend([resolved_start, *resolved_start.parents])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError("Could not locate MDNAC repo. Run inside the repo or set MDNAC_REPO_DIR.")


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import apply_instruction_notebook_overrides, bootstrap_notebook
from libs.instruction.trainer import InstructionTrainer, build_instruction_training_config

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(json.dumps(RUNTIME, indent=2, ensure_ascii=False))

{
  "repo_dir": "C:\\Users\\Admin\\Desktop\\MDNAC",
  "platform": "Windows",
  "platform_name": "Windows",
  "is_colab": false,
  "is_notebook": true,
  "python": "3.11.15",
  "cuda_available": false,
  "cuda_device_count": 0
}


In [ ]:
CONFIG_PATH = Path(os.environ.get("MDNAC_PROFILE_SPAN_CONFIG", REPO_DIR / "config" / "profile_span_completion.yaml")).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (REPO_DIR / CONFIG_PATH).resolve()

raw_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
CONFIG = build_instruction_training_config(REPO_DIR, raw_config)
CONFIG = apply_instruction_notebook_overrides(CONFIG)

instruction_paths = tuple(Path(path).expanduser().resolve() for path in CONFIG.instruction_jsonl)
protein_tokenizer_map = Path(CONFIG.sequence_tokenizer_map_path).expanduser().resolve()
expected_protein_tokenizer_map = (REPO_DIR / "tokenizer_map.json").resolve()
base_checkpoint = Path(CONFIG.base_checkpoint_path).expanduser().resolve()
artifact_dir = Path(CONFIG.artifact_dir).expanduser().resolve()
artifact_train_txt = artifact_dir / "train.txt"
artifact_tokenizer_map = artifact_dir / "tokenizer_map.json"

missing_instruction_paths = [path for path in instruction_paths if not path.is_file()]
assert not missing_instruction_paths, f"Missing span-completion instruction JSONL part: {missing_instruction_paths[0]}. Run notebook 01 first."
assert protein_tokenizer_map.is_file(), f"Missing stage-1 protein tokenizer map: {protein_tokenizer_map}"
assert protein_tokenizer_map == expected_protein_tokenizer_map, (
    "This run must reuse the root stage-1 protein tokenizer map. "
    f"Expected {expected_protein_tokenizer_map}, got {protein_tokenizer_map}."
)
assert artifact_tokenizer_map.is_file(), f"Missing profile-aware tokenizer map: {artifact_tokenizer_map}. Run notebook 02 first."
assert artifact_train_txt.is_file(), f"Missing profile-aware train.txt: {artifact_train_txt}. Run notebook 02 first."
if not base_checkpoint.is_file():
    raise FileNotFoundError(f"Base checkpoint not found: {base_checkpoint}. This notebook will not initialize from scratch.")
assert CONFIG.train_on_prompt is False
assert CONFIG.include_separator_in_loss is False
assert CONFIG.prompt_format == "compact_profile"

summary = {
    "config_path": str(CONFIG_PATH),
    "instruction_part_count": len(instruction_paths),
    "instruction_paths": [str(path) for path in instruction_paths],
    "protein_tokenizer_map": str(protein_tokenizer_map),
    "artifact_dir": str(artifact_dir),
    "artifact_tokenizer_map": str(artifact_tokenizer_map),
    "base_checkpoint": str(base_checkpoint),
    "checkpoint_dir": str(Path(CONFIG.output_dir).expanduser().resolve()),
    "prompt_format": CONFIG.prompt_format,
    "train_on_prompt": CONFIG.train_on_prompt,
    "include_separator_in_loss": CONFIG.include_separator_in_loss,
    "optimizer": CONFIG.optimizer_type,
    "learning_rate": CONFIG.learning_rate,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [3]:
from libs.core.pretrain.profiled import MDCProfileSequencePretrainArtifacts

artifacts = MDCProfileSequencePretrainArtifacts.from_directory(artifact_dir)
tokenizer_payload = json.loads(artifact_tokenizer_map.read_text(encoding="utf-8"))
sequence_tokenizer_payload = tokenizer_payload["sequence_tokenizer"]
source_tokenizer_map_path = sequence_tokenizer_payload.get("source_tokenizer_map_path")

assert sequence_tokenizer_payload["type"] == "sequence_bpe"
assert tokenizer_payload["compatibility"]["sequence_token_ids_preserved"] is True
assert source_tokenizer_map_path, "Profile-aware artifact did not record the source protein tokenizer map."
assert Path(source_tokenizer_map_path).resolve() == protein_tokenizer_map.resolve()

artifact_report = {
    "train_txt": str(artifact_train_txt),
    "tokenizer_map": str(artifact_tokenizer_map),
    "record_count": artifacts.record_count,
    "sequence_type": artifacts.sequence_type,
    "sequence_tokenizer_type": artifacts.sequence_tokenizer_type,
    "sequence_vocab_size": artifacts.sequence_vocab_size,
    "profile_vocab_size": artifacts.profile_vocab_size,
    "protein_tokenizer_loaded_from": source_tokenizer_map_path,
}
print(json.dumps(artifact_report, indent=2, ensure_ascii=False))


{
  "train_txt": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_span_completion_profile_pretrain\\train.txt",
  "tokenizer_map": "C:\\Users\\Admin\\Desktop\\MDNAC\\data\\compiled\\refseq_bacteria_span_completion_profile_pretrain\\tokenizer_map.json",
  "record_count": 100000,
  "sequence_type": "protein",
  "sequence_tokenizer_type": "sequence_bpe",
  "sequence_vocab_size": 256,
  "profile_vocab_size": 512,
  "protein_tokenizer_loaded_from": "C:\\Users\\Admin\\Desktop\\MDNAC\\tokenizer_map.json"
}


In [4]:
from libs.core.interfaces import IGNORE_INDEX
from libs.instruction.schema import iter_instruction_records

record = next(
    iter_instruction_records(
        instruction_paths,
        default_sequence_type=CONFIG.default_sequence_type,
        instruction_field=CONFIG.instruction_field,
        input_field=CONFIG.input_field,
        output_field=CONFIG.output_field,
        prompt_format=CONFIG.prompt_format,
    )
)
encoded = artifacts.encode_record(record)
fused = artifacts.build_fused_batch([encoded])
batch = artifacts.build_causal_lm_batch(
    [encoded],
    train_on_prompt=CONFIG.train_on_prompt,
    include_separator_in_loss=CONFIG.include_separator_in_loss,
)

sequence_start, sequence_end = [int(value) for value in fused.sequence_spans[0].tolist()]
expected_target_token_ids = fused.token_ids[0, sequence_start:sequence_end].tolist()
actual_target_labels = batch.labels[0, sequence_start - 1:sequence_end - 1].tolist()
prompt_labels = batch.labels[0, : max(0, sequence_start - 1)]
supervised_target_positions = [index + 1 for index, label in enumerate(batch.labels[0].tolist()) if label != IGNORE_INDEX]

assert int((prompt_labels != IGNORE_INDEX).sum().item()) == 0
assert actual_target_labels == expected_target_token_ids
assert supervised_target_positions == list(range(sequence_start, sequence_end))

masking_report = {
    "profile_preview": record.profile[:240],
    "target_missing_span": record.sequence,
    "target_residue_length": len(record.sequence),
    "sequence_start": sequence_start,
    "sequence_end": sequence_end,
    "prompt_label_count": int((prompt_labels != IGNORE_INDEX).sum().item()),
    "supervised_label_count": int((batch.labels != IGNORE_INDEX).sum().item()),
    "supervised_target_positions": supervised_target_positions[:16],
}
print(json.dumps(masking_report, indent=2, ensure_ascii=False))


{
  "profile_preview": "task protein span completion; labels protein sequence; description MULTISPECIES: scaffolding protein D [Bacteria].; organism Bacteria; keywords RefSeq; product scaffolding protein D; input mask_policy random_span; mask_start 14; mask_length",
  "target_missing_span": "LASIKLIQASAVLDLTEDDFDFLTSNKVWIATDRSRARRCVEACVYGT",
  "target_residue_length": 48,
  "sequence_start": 163,
  "sequence_end": 192,
  "prompt_label_count": 0,
  "supervised_label_count": 29,
  "supervised_target_positions": [
    163,
    164,
    165,
    166,
    167,
    168,
    169,
    170,
    171,
    172,
    173,
    174,
    175,
    176,
    177,
    178
  ]
}


In [ ]:
trainer = InstructionTrainer(CONFIG)
result = trainer.train()
print(json.dumps(result.to_dict(), indent=2, ensure_ascii=False))

In [ ]:
metrics_path = Path(result.metrics_history_path) if "result" in globals() else Path(CONFIG.output_dir) / "metrics_history.jsonl"


def safe_perplexity(loss: float | None) -> float | None:
    if loss is None or not math.isfinite(float(loss)):
        return None
    return math.exp(min(float(loss), 50.0))


rows = []
if metrics_path.is_file():
    for raw_line in metrics_path.read_text(encoding="utf-8").splitlines():
        if raw_line.strip():
            row = json.loads(raw_line)
            row["train_perplexity"] = safe_perplexity(row.get("train_loss"))
            row["val_perplexity"] = safe_perplexity(row.get("val_loss"))
            row["learning_rate"] = CONFIG.learning_rate
            rows.append(row)

print("metrics_history:", metrics_path)
if rows:
    for row in rows[-10:]:
        print(json.dumps(row, ensure_ascii=False))
else:
    print("No metrics rows yet. Run the training cell first.")

print("checkpoint_last:", Path(CONFIG.output_dir) / "checkpoint_last.pt")
print("checkpoint_best:", Path(CONFIG.output_dir) / "checkpoint_best.pt")
print("per_residue_accuracy: not available in the current BPE-token trainer; monitor target-token loss/perplexity here.")